In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import statsmodels.formula.api as smf
import seaborn as sns

In [60]:
mlflow.set_tracking_uri('http://localhost:5001')
mlflow.set_experiment("brecha_baseline_ols")

<Experiment: artifact_location=('/Users/santiagoquintana/Downloads/UNIANDES '
 'Local/M-IIND/2026-1/Analitic_comp/Proyecto2/ciencia_datos/ciencia_datosSQ/mlartifacts/3'), creation_time=1779739987540, experiment_id='3', last_update_time=1779739987540, lifecycle_stage='active', name='brecha_baseline_ols', tags={}, trace_location=None, workspace='default'>

In [62]:
data = pd.read_csv('../../EDA/data_finalSQ.csv')

# Create derived variables
data['hacinamiento'] = data['fami_personashogar'] / (data['fami_cuartoshogar'] + 1e-7)

print(f"✓ Datos cargados: {data.shape}")
print(f"✓ Variables derivadas creadas:")
print(f"  - hacinamiento: media={data['hacinamiento'].mean():.2f}")

print(f"
Variable target (brecha_mat_lec):")
print(f"  Media: {data['brecha_mat_lec'].mean():.2f}")
print(f"  Desv: {data['brecha_mat_lec'].std():.2f}")
print(f"  Rango: [{data['brecha_mat_lec'].min():.2f}, {data['brecha_mat_lec'].max():.2f}]")

print(f"
Variable moderadora (punt_tecnologia):")
print(f"  Valores: {sorted(data['punt_tecnologia'].unique())}")
print(f"  Distribución:")
print(data['punt_tecnologia'].value_counts().sort_index())

print(f"
Variable de estrato (fami_estratovivienda):")
print(f"  Rango: [{data['fami_estratovivienda'].min()}, {data['fami_estratovivienda'].max()}]")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    data,
    data['brecha_mat_lec'],
    test_size=0.2,
    random_state=42
)

# Crear DataFrame con variables necesarias
train_data = X_train.copy()
train_data['brecha_mat_lec'] = y_train.values

test_data = X_test.copy()
test_data['brecha_mat_lec'] = y_test.values

print(f"Train: {len(train_data)}, Test: {len(test_data)}")

Train: 68973, Test: 17244


In [ ]:
# Fórmula OLS con interacción
# Efecto principal de tecnología + moderación por estrato
formula = (
    "brecha_mat_lec ~ "
    "punt_tecnologia * C(fami_estratovivienda) + "  # interacción clave
    "punt_tecnologia * fami_educacionmadre + "      # interacción secundaria
    "fami_educacionpadre + "
    "fami_cuartoshogar + hacinamiento + "
    "fami_tienelavadora + "
    "C(cole_caracter) + C(cole_jornada) + "
    "cole_area_ubicacion + cole_naturaleza + estu_genero"
)

print(f"Fórmula OLS:")
print(f"  {formula}")
print(f"\nAjustando modelo...")

# Ajustar modelo OLS
modelo_ols = smf.ols(formula, data=train_data).fit()

print(f"✓ Modelo ajustado")
print(f"\nResumen del modelo:")
print(modelo_ols.summary())

Fórmula OLS:
  brecha_mat_lec ~ indice_tech * C(fami_estratovivienda) + indice_tech * fami_educacionmadre + fami_educacionpadre + fami_cuartoshogar + hacinamiento + fami_tienelavadora + C(cole_caracter) + C(cole_jornada) + cole_area_ubicacion + cole_naturaleza + estu_genero

Ajustando modelo...


PatsyError: Error evaluating factor: NameError: name 'hacinamiento' is not defined
    brecha_mat_lec ~ indice_tech * C(fami_estratovivienda) + indice_tech * fami_educacionmadre + fami_educacionpadre + fami_cuartoshogar + hacinamiento + fami_tienelavadora + C(cole_caracter) + C(cole_jornada) + cole_area_ubicacion + cole_naturaleza + estu_genero
                                                                                                                                           ^^^^^^^^^^^^

In [ ]:
# OLS Model Summary
print("\n" + "="*70)
print("OLS MODEL SUMMARY")
print("="*70)
print(modelo_ols.summary())

# Extract key statistics
print("\n" + "="*70)
print("KEY STATISTICS")
print("="*70)
print(f"F-statistic: {modelo_ols.fvalue:.4f}")
print(f"Prob (F-statistic): {modelo_ols.f_pvalue:.2e}")
print(f"R-squared: {modelo_ols.rsquared:.4f}")
print(f"Adj. R-squared: {modelo_ols.rsquared_adj:.4f}")
print(f"AIC: {modelo_ols.aic:.2f}")
print(f"BIC: {modelo_ols.bic:.2f}")
print(f"Observations: {modelo_ols.nobs}")
print(f"Degrees of Freedom: {modelo_ols.df_resid}")

In [ ]:
# Predicciones en test
y_pred_test = modelo_ols.get_prediction(test_data).summary_frame()
y_pred = y_pred_test['mean'].values

# Métricas
rmse = np.sqrt(mean_squared_error(y_test.values, y_pred))
mae = mean_absolute_error(y_test.values, y_pred)
r2 = r2_score(y_test.values, y_pred)
residuos = y_test.values - y_pred

print(f"\nMétricas en Test Set:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE: {mae:.4f}")
print(f"  R²: {r2:.4f}")
print(f"  Sesgo: {np.mean(residuos):.4f}")


Métricas en Test Set:
  RMSE: 8.0076
  MAE: 6.2602
  R²: 0.0512
  Sesgo: 0.1262


In [ ]:
# Extraer coeficientes de interacción clave
print(f"\n" + "="*70)
print(f"EFECTO MODERADOR: punt_tecnologia × fami_estratovivienda")
print(f"="*70)

# Mostrar coeficientes relacionados
coefs = modelo_ols.params
pvals = modelo_ols.pvalues

for coef_name in coefs.index:
    if 'punt_tecnologia' in coef_name and 'estrato' in coef_name.lower():
        coef_val = coefs[coef_name]
        pval = pvals[coef_name]
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"\n{coef_name}:")
        print(f"  Coeficiente: {coef_val:.6f} {sig}")
        print(f"  p-valor: {pval:.6f}")
        print(f"  Interpretación: Por cada unidad de tecnología adicional,")
        print(f"                  la brecha cambia {coef_val:.4f} puntos en este estrato")

print(f"\n*** p<0.001, ** p<0.01, * p<0.05")


EFECTO MODERADOR: indice_tech × fami_estratovivienda

indice_tech:C(fami_estratovivienda)[T.2.0]:
  Coeficiente: 0.132886 
  p-valor: 0.142027
  Interpretación: Por cada unidad de tecnología adicional,
                  la brecha cambia 0.1329 puntos en este estrato

indice_tech:C(fami_estratovivienda)[T.3.0]:
  Coeficiente: 0.168388 
  p-valor: 0.256659
  Interpretación: Por cada unidad de tecnología adicional,
                  la brecha cambia 0.1684 puntos en este estrato

indice_tech:C(fami_estratovivienda)[T.4.0]:
  Coeficiente: 0.511543 
  p-valor: 0.181931
  Interpretación: Por cada unidad de tecnología adicional,
                  la brecha cambia 0.5115 puntos en este estrato

indice_tech:C(fami_estratovivienda)[T.5.0]:
  Coeficiente: 2.028815 **
  p-valor: 0.001444
  Interpretación: Por cada unidad de tecnología adicional,
                  la brecha cambia 2.0288 puntos en este estrato

indice_tech:C(fami_estratovivienda)[T.6.0]:
  Coeficiente: -0.697337 
  p-valor: 0.4484

In [ ]:
# Registrar en MLflow
with mlflow.start_run(run_name="OLS_baseline_moderacion"):
    # Parámetros del modelo
    mlflow.log_params({
        "modelo_tipo": "OLS",
        "n_features": len(coefs),
        "interaccion_principal": "punt_tecnologia × C(fami_estratovivienda)",
        "interaccion_secundaria": "punt_tecnologia × fami_educacionmadre"
    })
    
    # Métricas principales
    mlflow.log_metrics({
        "test_rmse": rmse,
        "test_mae": mae,
        "test_r2": r2,
        "train_r2": modelo_ols.rsquared,
        "adj_r2": modelo_ols.rsquared_adj,
        "f_statistic": modelo_ols.fvalue,
    })
    
    print(f"✓ Modelo registrado en MLflow")
    print(f"  Run name: OLS_baseline_moderacion")
    print(f"  Métricas: RMSE={rmse:.4f}, R²={r2:.4f}")


✓ Modelo registrado en MLflow
  Run name: OLS_baseline_moderacion
  Métricas: RMSE=8.0076, R²=0.0512
🏃 View run OLS_baseline_moderacion at: http://localhost:5001/#/experiments/3/runs/a35354f37a474014b8d89c5045a575d8
🧪 View experiment at: http://localhost:5001/#/experiments/3


In [ ]:
# Gráficas
# 1. Real vs Predicho
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test.values, y_pred, alpha=0.3, s=10, color='steelblue')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Predicción perfecta')
ax.set_xlabel('Brecha real')
ax.set_ylabel('Brecha predicha')
ax.set_title(f'OLS: Real vs Predicho (R²={r2:.4f})')
ax.legend()
ax.grid(alpha=0.3)
mlflow.log_figure(fig, "01_ols_real_vs_predicho.png")
plt.close(fig)

# 2. Residuos
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(y_pred, residuos, alpha=0.3, s=10, color='steelblue')
ax.axhline(0, color='r', linestyle='--', linewidth=2)
ax.set_xlabel('Predicciones')
ax.set_ylabel('Residuos')
ax.set_title('Residuos vs Predicciones')
ax.grid(alpha=0.3)
mlflow.log_figure(fig, "02_ols_residuos.png")
plt.close(fig)

# 3. Coeficientes
fig, ax = plt.subplots(figsize=(10, 6))
# Top 15 coeficientes por magnitud (excluyendo intercept)
coefs_sorted = coefs.drop('Intercept').abs().nlargest(15)
ax.barh(range(len(coefs_sorted)), coefs_sorted.values)
ax.set_yticks(range(len(coefs_sorted)))
ax.set_yticklabels(coefs_sorted.index, fontsize=9)
ax.set_xlabel('Coeficiente (valor absoluto)')
ax.set_title('Top 15 Coeficientes más grandes (OLS)')
ax.grid(alpha=0.3, axis='x')
plt.tight_layout()
mlflow.log_figure(fig, "03_ols_coeficientes.png")
plt.close(fig)

print(f"✓ 3 gráficas registradas en MLflow")

✓ 3 gráficas registradas en MLflow


In [ ]:
# GRÁFICA 4: Efecto moderador - Regresión estimada por estrato
# Mostrar cómo punt_tecnologia afecta la brecha según fami_estratovivienda

fig, ax = plt.subplots(figsize=(10, 6))

# Obtener estratos únicos
estratos = sorted(test_data['fami_estratovivienda'].unique())
colors = plt.cm.Set2(np.linspace(0, 1, len(estratos)))

# Para cada estrato, graficar scatter + línea estimada
for idx_estrato, estrato in enumerate(estratos):
    mask = test_data['fami_estratovivienda'] == estrato
    
    # Scatter del estrato
    ax.scatter(test_data.loc[mask, 'punt_tecnologia'],
              y_test.values[mask],
              alpha=0.4, s=30, label=f'Estrato {int(estrato)}',
              color=colors[idx_estrato])
    
    # Línea de regresión estimada para este estrato
    tech_range = np.linspace(test_data['punt_tecnologia'].min(),
                             test_data['punt_tecnologia'].max(), 50)
    
    # Crear datos para predicción (mismo estrato)
    pred_data = test_data.loc[mask].iloc[:1].copy()
    pred_data = pd.concat([pred_data] * len(tech_range), ignore_index=True)
    pred_data['punt_tecnologia'] = tech_range
    pred_data['fami_estratovivienda'] = estrato
    
    # Predecir
    y_pred_estrato = modelo_ols.get_prediction(pred_data).summary_frame()['mean'].values
    
    # Graficar línea
    ax.plot(tech_range, y_pred_estrato, linewidth=2.5,
           color=colors[idx_estrato], label=f'Regresión E{int(estrato)}')

ax.set_xlabel('Índice de Tecnología (computador + internet)', fontsize=11)
ax.set_ylabel('Brecha (Matemática - Lectura Crítica)', fontsize=11)
ax.set_title('OLS: Efecto Moderador de Estrato en la Brecha', fontsize=12, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
plt.tight_layout()
mlflow.log_figure(fig, "04_ols_moderacion.png")
plt.close(fig)

print(f"✓ Gráfica 04_ols_moderacion.png registrada")
print(f"  Muestra cómo el efecto de tecnología varía según estrato socioeconómico")


✓ Gráfica 04_ols_moderacion.png registrada
  Muestra cómo el efecto de tecnología varía según estrato socioeconómico


In [ ]:
# Guardar modelo
import joblib
import json
import os

os.makedirs("ols_brecha", exist_ok=True)
joblib.dump(modelo_ols, "ols_brecha/modelo_ols_baseline.pkl")

# Guardar metadata
metadata = {
    "tipo_modelo": "OLS",
    "formula": formula,
    "test_rmse": float(rmse),
    "test_r2": float(r2),
    "train_r2": float(modelo_ols.rsquared),
    "adj_r2": float(modelo_ols.rsquared_adj),
    "n_obs": len(train_data),
    "interaccion_clave": "punt_tecnologia × C(fami_estratovivienda)"
}
with open("ols_brecha/metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✓ Modelo OLS guardado en ols_brecha/")
print(f"✓ Metadata guardada")
print(f"\n" + "="*70)
print(f"CONCLUSIÓN: Este modelo serve como baseline interpretable.")
print(f"Comparar RMSE y R² contra el Modelo 1 (red neuronal) para ver")
print(f"si la complejidad de la red mejora el ajuste.")
print(f"="*70)

✓ Modelo OLS guardado en ols_brecha/
✓ Metadata guardada

CONCLUSIÓN: Este modelo serve como baseline interpretable.
Comparar RMSE y R² contra el Modelo 1 (red neuronal) para ver
si la complejidad de la red mejora el ajuste.
